# Calculadora Von Neumann optimizada — Interfaz para Colab

Ejecuta la celda de abajo (Shift+Enter). Usa solo `dataclasses` (estándar de Python) e `ipywidgets`, que viene preinstalado en Google Colab — no requiere instalar nada.

In [1]:
# Librerias usadas: dataclasses (estandar) e ipywidgets (preinstalada en Colab)
import ipywidgets as widgets
from IPython.display import display, clear_output
from dataclasses import dataclass, field


# MEMORIA PRINCIPAL
@dataclass
class Memoria:
    historial: list = field(default_factory=list)

    def escribir(self, direccion, valor):
        self.historial.append({"direccion": direccion, "valor": valor})


# REGISTROS DE LA CPU
@dataclass
class Registros:
    instruccion: str = None
    operando_a: float = 0.0
    operando_b: float = 0.0
    resultado: float = 0.0


# ALU
class ALU:
    @staticmethod
    def sumar(a, b):
        return a + b

    @staticmethod
    def restar(a, b):
        return a - b

    @staticmethod
    def multiplicar(a, b):
        return a * b

    @staticmethod
    def dividir(a, b):
        if b == 0:
            raise ZeroDivisionError("La ALU no puede dividir entre cero")
        return a / b


# UNIDAD DE CONTROL (tabla de despacho)
class UnidadDeControl:
    def __init__(self):
        self.alu = ALU()
        self.tabla_instrucciones = {
            "1": ("SUMA", self.alu.sumar),
            "2": ("RESTA", self.alu.restar),
            "3": ("MULTIPLICACION", self.alu.multiplicar),
            "4": ("DIVISION", self.alu.dividir),
        }

    def decodificar(self, opcode):
        instruccion = self.tabla_instrucciones.get(opcode)
        if instruccion is None:
            raise ValueError(f"Instruccion no valida: {opcode!r}")
        return instruccion


# CPU: orquesta el ciclo Fetch-Decode-Execute-WriteBack
class CPU:
    def __init__(self):
        self.registros = Registros()
        self.memoria = Memoria()
        self.control = UnidadDeControl()
        self.contador_programa = 0

    def ejecutar_ciclo(self, opcode, a, b, log_callback):
        log_callback(f"--- CICLO DE INSTRUCCION #{self.contador_programa} ---")

        self.registros.instruccion = opcode
        self.registros.operando_a = a
        self.registros.operando_b = b
        log_callback(f"FETCH      -&gt; instruccion={opcode}, reg_a={a}, reg_b={b}")

        nombre_op, funcion = self.control.decodificar(opcode)
        log_callback(f"DECODE     -&gt; operacion identificada: {nombre_op}")

        try:
            resultado = funcion(self.registros.operando_a, self.registros.operando_b)
            self.registros.resultado = resultado
            log_callback(f"EXECUTE    -&gt; ALU calculo: {resultado}")
        except ZeroDivisionError as err:
            log_callback(f"EXECUTE    -&gt; ERROR en la ALU: {err}")
            self.registros.resultado = None

        self.memoria.escribir(self.contador_programa, self.registros.resultado)
        log_callback(
            f"WRITE-BACK -&gt; guardado en memoria[{self.contador_programa}] = {self.registros.resultado}"
        )
        self.contador_programa += 1
        return self.registros.resultado

# INTERFAZ (widgets)
cpu = CPU()

titulo = widgets.HTML("<h2>Calculadora Von Neumann - Version optimizada</h2>")

entrada_a = widgets.FloatText(description="Reg A:", value=0)
entrada_b = widgets.FloatText(description="Reg B:", value=0)
operacion = widgets.Dropdown(
    options=[("Sumar (+)", "1"), ("Restar (-)", "2"),
             ("Multiplicar (x)", "3"), ("Dividir (/)", "4")],
    description="Instruccion:",
)

boton_ejecutar = widgets.Button(description="Ejecutar ciclo", button_style="success")
boton_limpiar = widgets.Button(description="Limpiar consola", button_style="warning")

panel_registros = widgets.HTML()
consola = widgets.Output(layout=widgets.Layout(
    border="1px solid black", height="220px", overflow_y="auto"
))
panel_memoria = widgets.Output(layout=widgets.Layout(
    border="1px solid gray", height="120px", overflow_y="auto"
))


def actualizar_panel_registros():
    r = cpu.registros
    panel_registros.value = (
        "<b>Estado de los registros</b><br>"
        f"Instruccion: {r.instruccion}<br>"
        f"Reg A: {r.operando_a}<br>"
        f"Reg B: {r.operando_b}<br>"
        f"Reg Resultado: {r.resultado}<br>"
        f"Program Counter: {cpu.contador_programa}"
    )


def actualizar_memoria():
    with panel_memoria:
        clear_output()
        for pos in cpu.memoria.historial:
            print(f"[{pos['direccion']}] = {pos['valor']}")


def log_en_consola(texto):
    with consola:
        print(texto)


def al_hacer_click_ejecutar(_):
    cpu.ejecutar_ciclo(operacion.value, entrada_a.value, entrada_b.value, log_en_consola)
    actualizar_panel_registros()
    actualizar_memoria()


def al_hacer_click_limpiar(_):
    with consola:
        clear_output()


boton_ejecutar.on_click(al_hacer_click_ejecutar)
boton_limpiar.on_click(al_hacer_click_limpiar)

actualizar_panel_registros()

entradas = widgets.VBox([entrada_a, entrada_b, operacion,
                          widgets.HBox([boton_ejecutar, boton_limpiar]),
                          panel_registros])
salida = widgets.VBox([widgets.HTML("<b>Ciclo Fetch-Decode-Execute-WriteBack</b>"), consola,
                        widgets.HTML("<b>Memoria (historial)</b>"), panel_memoria])

display(titulo, widgets.HBox([entradas, salida]))


HTML(value='<h2>Calculadora Von Neumann - Version optimizada</h2>')